In [64]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  Cell 1 – Imports & global config                                    ║
# ╚═══════════════════════════════════════════════════════════════════════╝
import os, math, random, warnings
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
import torchvision.transforms as T
from torchvision import models

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_CLASSES = 3          # we have exactly 3 score bands (1-3 → 0-2)
print(f"✅ Torch {torch.__version__}  CUDA={torch.cuda.is_available()}")


✅ Torch 2.7.0+cpu  CUDA=False


In [65]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  Cell 2 – Allowed augmentation & preprocessing                        ║
# ╚═══════════════════════════════════════════════════════════════════════╝
IMG_SIZE = 224   # upscale 200×200 → 224×224 for ResNet

class BoldLines:
    """
    Randomly dilate strokes up to +40 % thickness.
    Works on a PIL Image (L mode, white foreground on black).
    """
    def __init__(self, p=0.6, max_pct=0.4):
        self.p        = p
        self.max_iter = int(max_pct * 10) or 1  # iterations 1–4 roughly 10–40 %

    def __call__(self, img):
        if np.random.rand() > self.p:
            return img
        import cv2
        arr    = np.array(img)
        ksize  = 3
        kernel = np.ones((ksize, ksize), np.uint8)
        iters  = np.random.randint(1, self.max_iter + 1)
        arr    = cv2.dilate(arr, kernel, iterations=iters)
        return Image.fromarray(arr)

train_tf = T.Compose([
    # 1. shift around centre  (≤10 % width / height)
    T.RandomAffine(degrees=0,
                   translate=(.1, .1),
                   fill=0),
    # 2. tiny rotation ±2–5 °
    T.RandomAffine(degrees=(-5, 5),
                   translate=(0, 0),
                   fill=0),
    # 3. bold strokes
    BoldLines(p=0.7, max_pct=0.4),
    # 4. local “patch” noise  (RandomErasing  acts on tensor)
    T.ToTensor(),               # → 1×H×W  (0–1)
    T.Lambda(lambda t: t.repeat(3,1,1)),   # to 3-channel
    T.RandomErasing(p=0.25,
                    scale=(.02, .07),
                    ratio=(.5, 2.0),
                    value=0),
    # final resize
    T.Resize((IMG_SIZE, IMG_SIZE)),
])

val_tf = T.Compose([
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Resize((IMG_SIZE, IMG_SIZE)),
])

print("✅ Augmentation pipeline ready")


✅ Augmentation pipeline ready


In [66]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  Cell 3 – Dataset wrapper  (filtered to scores 1-3 only)             ║
# ╚═══════════════════════════════════════════════════════════════════════╝
class DrawingDataset(Dataset):
    """
    CSV columns required: child_id, shape, score
    Only rows whose score ∈ {1,2,3} are kept.
    """
    def __init__(self, csv_path, img_dir, shape_id, split="train"):
        super().__init__()
        df = pd.read_csv(csv_path)
        df = df[(df["shape"] == shape_id) & (df["score"].isin([1, 2, 3]))]

        self.paths, self.labels = [], []
        for _, row in df.iterrows():
            path = os.path.join(img_dir, f"{int(row['child_id'])}.png")
            if os.path.exists(path):
                self.paths.append(path)
                self.labels.append(int(row["score"]) - 1)   # 1-3 → 0-2

        self.tf = train_tf if split == "train" else val_tf
        print(f"{split.capitalize()} set: {len(self.paths)} images")

    def __len__(self):  return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("L")
        x   = self.tf(img)
        y   = self.labels[idx]
        return x, y


In [67]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  Cell 4 – Balanced loaders (≈250 samples per class, on-the-fly)       ║
# ╚═══════════════════════════════════════════════════════════════════════╝
TARGET_PER_CLASS = 250          # 250 × 3 classes  →  750 samples / epoch


class BalancedAugSubset(Dataset):
    """
    Wraps a Subset of DrawingDataset and, at __getitem__, returns a randomly
    chosen example *within a fixed class*, so that each epoch exposes
    TARGET_PER_CLASS samples per class.

    • Keeps the original transform pipeline (shift, tiny rotate, bold, noise).
    • No extra storage – every access generates a fresh augmentation.
    """
    def __init__(self, subset: torch.utils.data.Subset,
                 n_classes: int = N_CLASSES,
                 target_per_class: int = TARGET_PER_CLASS):
        self.subset           = subset
        self.n_classes        = n_classes
        self.target_per_class = target_per_class

        # map: class → list of base-dataset indices inside this subset
        self.cls_to_indices = [[] for _ in range(n_classes)]
        for base_idx in subset.indices:
            label = subset.dataset.labels[base_idx]    # 0,1,2
            self.cls_to_indices[label].append(base_idx)

    def __len__(self):
        return self.n_classes * self.target_per_class

    def __getitem__(self, idx):
        cls_id   = idx // self.target_per_class             # 0,1,2
        base_ids = self.cls_to_indices[cls_id]
        base_idx = random.choice(base_ids)                  # random example
        return self.subset.dataset[base_idx]                # applies transform


def make_loaders(csv_path, img_dir, shape_id, batch=32):
    """
    1. Loads DrawingDataset filtered to scores 1-3 only
    2. 80 / 20 split  →  train / val
    3. Wraps the *train* split with BalancedAugSubset
       so every epoch shows ≈ TARGET_PER_CLASS samples per class
    4. Computes class weights on the *original* (pre-aug) train set
    """
    full_ds = DrawingDataset(csv_path, img_dir, shape_id, split="train")

    # ── 80 / 20 split ────────────────────────────────────────────────────
    n_total      = len(full_ds)
    n_train      = int(0.8 * n_total)
    n_val        = n_total - n_train
    base_tr, vl_ds = random_split(
        full_ds,
        [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED)
    )

    # ── balanced, augmented wrapper for the *training* subset ────────────
    tr_ds = BalancedAugSubset(base_tr, n_classes=N_CLASSES,
                              target_per_class=TARGET_PER_CLASS)

    # ── class weights (compute on original counts, length == N_CLASSES) ─
    tr_labels = [full_ds.labels[i] for i in base_tr.indices]
    counts    = Counter(tr_labels)
    max_cnt   = max(counts.values())
    cls_wt    = torch.tensor(
        [max_cnt / counts.get(c, 1) for c in range(N_CLASSES)],
        dtype=torch.float32,
        device=DEVICE
    )

    # ── loaders ─────────────────────────────────────────────────────────
    tr_loader = DataLoader(tr_ds,
                           batch_size=batch,
                           shuffle=True,           # balanced already
                           num_workers=0)

    vl_loader = DataLoader(vl_ds,
                           batch_size=batch,
                           shuffle=False,
                           num_workers=0)

    # ── report -----------------------------------------------------------------
    print(f"\n🖼️  Original train images: {len(base_tr)} "
          f"(class distrib {dict(counts)})")
    print(f"🖼️  Augmented train samples/epoch: "
          f"{len(tr_ds)}  ({TARGET_PER_CLASS} × {N_CLASSES})")
    print(f"🖼️  Validation images: {len(vl_ds)}")

    return tr_loader, vl_loader, cls_wt


In [68]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  Cell 5 –  LoReFT intervention (low-rank residual tweak)             ║
# ╚═══════════════════════════════════════════════════════════════════════╝
class LoReFT(nn.Module):
    """
    LoReFT from the ReFT paper – learned low-rank map U·Vᵀ applied in residual space.
    h_out = h_in + U·Vᵀ·h_in.  Only U,V are trainable (r ≪ d).
    """
    def __init__(self, hidden_dim, rank=16):
        super().__init__()
        self.U = nn.Parameter(torch.randn(hidden_dim, rank) * 0.02)
        self.V = nn.Parameter(torch.randn(hidden_dim, rank) * 0.02)

    def forward(self, h):               # h: (B, d)
        return h + (h @ self.V) @ self.U.T


In [74]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  Cell 6 – Build ViT-B/16  +  LoReFT  +  classifier                    ║
# ╚═══════════════════════════════════════════════════════════════════════╝
import timm                      # <-- new import

def build_vit_model(n_classes: int,
                    rank: int = 64,
                    unfreeze_blocks: tuple = (10, 11),   # last 2 blocks
                    p_dropout: float = 0.3):
    """
    vit_base_patch16_224  (embed_dim=768)  →  LoReFT(rank=64)  →  Dropout →  FC
    Only transformer blocks listed in `unfreeze_blocks` are trainable.
    """
    vit = timm.create_model('vit_base_patch16_224',
                            pretrained=True,
                            num_classes=0,       # output CLS token only
                            global_pool='token')

    # freeze everything
    for p in vit.parameters():
        p.requires_grad = False
    # un-freeze last transformer blocks
    for idx in unfreeze_blocks:
        for p in vit.blocks[idx].parameters():
            p.requires_grad = True

    feat_dim = vit.embed_dim            # 768
    model = nn.Sequential(
        vit,                            # (B, 768)
        LoReFT(feat_dim, rank=rank),    # (B, 768)
        nn.Dropout(p_dropout),
        nn.Linear(feat_dim, n_classes)
    ).to(DEVICE)

    return model


# build
model = build_vit_model(N_CLASSES)
print(f"Trainable parameters: "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Trainable parameters: 14,276,355


In [75]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  Cell 7 – Training & evaluation helpers                              ║
# ╚═══════════════════════════════════════════════════════════════════════╝
def accuracy(outputs, labels):
    return (outputs.argmax(1) == labels).float().mean().item()

def run_epoch(model, loader, loss_fn, opt=None):
    train = opt is not None
    model.train() if train else model.eval()
    total_loss, total_acc = 0., 0.
    for x,y in (tqdm(loader, leave=False) if train else loader):
        x,y = x.to(DEVICE), y.to(DEVICE)
        if train:
            opt.zero_grad()
        out  = model(x)
        loss = loss_fn(out, y)
        if train:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.)
            opt.step()
        total_loss += loss.item() * x.size(0)
        total_acc  += (out.argmax(1)==y).sum().item()
    n = len(loader.dataset)
    return total_loss/n, total_acc/n

In [78]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  Cell 8 – Fit loop (ViT; 3-tier LRs; label smoothing; cosine sched)   ║
# ╚═══════════════════════════════════════════════════════════════════════╝
def fit(model,
        tr_loader,
        vl_loader,
        cls_wts,
        epochs: int = 30,
        lr: float = 3e-4,
        patience: int = 8):

    # ── loss (label-smoothing) ──────────────────────────────────────────
    loss_fn = nn.CrossEntropyLoss(weight=cls_wts,
                                  label_smoothing=0.10)

    # ── optimiser: blocks.10, blocks.11, head/LoReFT ────────────────────
    p_blk10, p_blk11, p_head = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "blocks.11" in n:          # last ViT block
            p_blk11.append(p)
        elif "blocks.10" in n:        # penultimate block
            p_blk10.append(p)
        else:                         # LoReFT + final FC
            p_head.append(p)

    opt = optim.AdamW(
        [{"params": p_blk10, "lr": lr * 0.05},
         {"params": p_blk11, "lr": lr * 0.10},
         {"params": p_head,  "lr": lr}],
        weight_decay=1e-4
    )

    # ── cosine annealing scheduler ──────────────────────────────────────
    sched = optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=epochs, eta_min=1e-6)

    # ── training loop ───────────────────────────────────────────────────
    best_vl, patience_ctr = math.inf, 0
    hist = {"tr_loss": [], "vl_loss": [], "vl_acc": []}

    for ep in range(1, epochs + 1):
        tr_loss, _      = run_epoch(model, tr_loader, loss_fn, opt)
        vl_loss, vl_acc = run_epoch(model, vl_loader, loss_fn, opt=None)
        sched.step()

        hist["tr_loss"].append(tr_loss)
        hist["vl_loss"].append(vl_loss)
        hist["vl_acc" ].append(vl_acc)

        print(f"E{ep:02d}  train {tr_loss:.3f}  |  val {vl_loss:.3f}  "
              f"|  acc {vl_acc:.3f}")

        # save best model
        if vl_loss < best_vl - 1e-4:
            best_vl, patience_ctr = vl_loss, 0
            torch.save(model.state_dict(),
                       "best_vit_loreft.pth")
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print("⏹️  Early stop")
                break
    return hist


In [79]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  Cell 9 – Kick off training                                          ║
# ╚═══════════════════════════════════════════════════════════════════════╝
CSV_PATH = r"C:\Users\yozev\PycharmProjects\finetuning\child_shape_scores.csv"
IMG_DIR  = r"C:\Users\yozev\OneDrive\Desktop\Shapes2\shape2"
SHAPE_ID = 2
BATCH    = 32

tr_loader, vl_loader, cls_wts = make_loaders(CSV_PATH, IMG_DIR, SHAPE_ID, batch=BATCH)
print("🏁 Starting …\n")
history = fit(model, tr_loader, vl_loader, cls_wts, epochs=30, lr=3e-4, patience=8)

Train set: 286 images

🖼️  Original train images: 228 (class distrib {1: 127, 0: 57, 2: 44})
🖼️  Augmented train samples/epoch: 750  (250 × 3)
🖼️  Validation images: 58
🏁 Starting …



  0%|          | 0/24 [00:00<?, ?it/s]

E01  train 0.736  |  val 1.042  |  acc 0.466


  0%|          | 0/24 [00:00<?, ?it/s]

E02  train 0.715  |  val 1.027  |  acc 0.379


...

E03  train 0.673  |  val 0.983  |  acc 0.397


  0%|          | 0/24 [00:00<?, ?it/s]

E04  train 0.607  |  val 1.012  |  acc 0.362


  0%|          | 0/24 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  Cell 10 – Training curves                                           ║
# ╚═══════════════════════════════════════════════════════════════════════╝
plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.plot(history["tr_loss"], label="train"); plt.plot(history["vl_loss"], label="val"); plt.title("Loss"); plt.legend()
plt.subplot(1,3,2); plt.plot(history["vl_acc"]); plt.title("Val-acc"); plt.ylim(0,1)
plt.subplot(1,3,3); plt.plot([g["lr"] for g in optim.AdamW(model.parameters()).param_groups]); plt.title("LR (initial)")
plt.tight_layout(); plt.show()

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  Cell 11 – Confusion counts & per-class accuracy                      ║
# ╚═══════════════════════════════════════════════════════════════════════╝
model.load_state_dict(torch.load("best_loreft_resnet18.pth"))
model.eval()

all_preds, all_lbls = [], []
with torch.no_grad():
    for x,y in vl_loader:
        out = model(x.to(DEVICE))
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_lbls.extend(y.tolist())

cm = Counter([(l,p) for l,p in zip(all_lbls, all_preds)])
print("Confusion (label→pred counts):", cm)

overall = sum(1 for l,p in zip(all_lbls, all_preds) if l==p) / len(all_lbls)
print(f"Overall accuracy: {overall:.3f}")

for c in sorted(set(all_lbls)):
    tp  = cm.get((c,c),0)
    tot = sum(cm.get((c,p),0) for p in range(3))
    print(f"Class {c}: {tp}/{tot} ({tp/tot if tot else 0:.3f})")